# Notebook 06a — Signing in: OAuth/PKCE and the /callback route

**ATLAS: Aligned Three-Layer Architecture for Semantics**
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

You deployed the UI in notebook 06 and the distributions are serving. Now a banker opens
the URL and clicks sign in — and two questions come up that the deploy step alone does not
answer:

> How does the user actually authenticate, when this is a static site with no server and
> no client secret? And why did the `/callback` page return an error until we changed the
> CloudFront routing?

These two questions are really one story. The sign-in flow ends by redirecting the browser
to a `/callback` page, and that page only works if CloudFront serves it correctly while
preserving the authorization code in the URL. Get the auth flow right but the routing
wrong and login fails at the last step; get the routing right but skip PKCE and the flow
is insecure for a secretless client. This notebook teaches both as the single, correct
design they are.


## The concept: a secretless sign-in, and a route that keeps the code

### Why authorization-code with PKCE

The two UIs are static single-page apps served from S3 behind CloudFront. There is no
server side and no client secret — the Cognito app client is created with
`generateSecret: false`. That rules out any flow that authenticates the token exchange
with a secret, and it makes the bare authorization-code flow unsafe on its own: an
intercepted authorization code could be replayed by anyone.

PKCE (Proof Key for Code Exchange) is the spec's answer for exactly this case, a public
client with no secret. The flow generates a high-entropy random `code_verifier`, derives
its SHA-256 `code_challenge`, and sends only the challenge on the redirect to the hosted
UI. When the browser later exchanges the authorization code for tokens, it sends the
original verifier. Cognito rejects any exchange whose verifier does not hash to the
challenge it saw on the authorize request. That binding is what defeats code interception:
a stolen code is useless without the verifier, which never left the browser's session
storage. PKCE here is not a workaround for a missing secret — it is the correct way to do
authorization-code for a client that has no secret to begin with.

The flow, end to end: the app redirects to the hosted UI's `/oauth2/authorize` with the
client id, the `/callback` redirect URI, the scopes, and the S256 challenge; the user
authenticates; Cognito redirects back to `/callback?code=...`; the callback page reads the
code, sends it with the stored verifier to `/oauth2/token`, and stores the returned access
token under the key the Apollo client reads. The session is then restored from that token.

### Why a path-preserving rewrite, not a 403 fallback

That `/callback` redirect is where the routing problem lives. The apps are built with
Next.js `output: "export"` and `trailingSlash: true`, so every route is its own document:
`/callback` is the file `out/callback/index.html`, not a server route. The CloudFront
origin is S3 over an Origin Access Control REST endpoint, and that origin returns 403 for
any path with no exact object. The path `/callback` has no exact object, so without
intervention Cognito's redirect lands on a 403 and login dies at the final step.

The obvious fix is the wrong one. A naive single-page-app fallback — map any 403 to
`/index.html` — would serve the app, but the root document is not the callback handler
(these apps export per-route HTML), and worse, rewriting to `/index.html` discards the
path while the browser still carries `?code=...` only against the original URL. The code
would be dropped and the exchange would never happen.

The correct fix is a viewer-request CloudFront Function that rewrites the clean route to
its own per-route document and leaves the query string alone. The query string is not part
of `request.uri`, so CloudFront forwards `?code=...` unchanged to the callback document,
where the client-side handler reads it. The function's guard order is deliberate: the bare
root passes through to the default root object; a trailing-slash path gets `index.html`
appended; a path whose last segment has no dot is treated as a route and gets
`/index.html` appended; anything with a file extension, like `/_next/static/x.js`, passes
through untouched so real assets still load. Path-preserving, code-preserving, asset-safe —
that is why this function and not a fallback.


## Show me: the real code on both sides

The flow is in `apps/shared/auth/` (the redirect and the token exchange) and the routing
is one CloudFront Function in the CDK. The cells below read the actual source so the lesson
stays tied to what runs.


In [ ]:
import os

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))  # use-case-applications/

def show(path, start, end, title):
    full = os.path.join(REPO, path)
    print(f"# {title}  ({path}:{start}-{end})\n")
    with open(full) as f:
        lines = f.read().splitlines()
    for n in range(start, min(end, len(lines)) + 1):
        print(f"{n:4} {lines[n-1]}")
    print()


### The PKCE helpers — verifier in, S256 challenge out

The verifier is 48 random bytes, base64url-encoded, held in session storage across the
redirect; the challenge is its SHA-256, derived with the browser's WebCrypto.


In [ ]:
show("apps/shared/auth/pkce.ts", 25, 46, "pkce.ts — createAndStoreVerifier + deriveChallenge")

### The redirect — what sign-in actually sends to the hosted UI

The authorize URL carries the challenge (not the verifier) and names `/callback` as the
redirect target. The verifier stays in the browser until the exchange.


In [ ]:
show("apps/shared/auth/use-auth.ts", 84, 101, "use-auth.ts — signIn() builds the /oauth2/authorize redirect")

### The exchange — code plus verifier in, access token out

On the `/callback` side the handler reads the code, pulls the stored verifier, and posts
both to `/oauth2/token`. Cognito checks the verifier against the challenge it saw.


In [ ]:
show("apps/shared/auth/use-auth.ts", 174, 200, "use-auth.ts — exchangeCodeForToken()")

### The route — the viewer-request rewrite that keeps the code

This is the whole fix: map a clean route to its per-route `index.html`, pass assets and the
query string through untouched.


In [ ]:
show("cdk/lib/constructs/cloudfront.ts", 50, 67, "cloudfront.ts — SpaRewriteFn (viewer-request)")

## Verify it: the route resolves, the redirect is well-formed

The decisive check is that `/callback` now returns 200 rather than 403, and that the
authorize URL the app builds is well-formed against the deployed Cognito domain. The cell
needs the deployed CloudFront URL and Cognito values; without a deployment it describes the
expected result instead (the same precondition the build and acceptance notebooks have).
Signing in fully is a human click-through — this confirms the surfaces that make it work.


In [ ]:
import json, subprocess, urllib.request, urllib.parse

def stack_outputs(stack="AtlasWorkshop2", region="us-east-1"):
    p = subprocess.run(["aws","cloudformation","describe-stacks","--stack-name",stack,
                        "--region",region,"--query","Stacks[0].Outputs","--output","json"],
                       capture_output=True, text=True)
    if p.returncode != 0:
        return {}
    return {o["OutputKey"]: o["OutputValue"] for o in json.loads(p.stdout or "[]")}

out = stack_outputs()
ui = out.get("WholesaleUiUrl", "")
domain = out.get("CognitoHostedUiDomain", "")
client = out.get("CognitoUserPoolWebClientId", "")

if ui:
    def status(url):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, method="GET"), timeout=10) as r:
                return r.status
        except urllib.error.HTTPError as e:
            return e.code
        except Exception as e:
            return f"err: {e}"
    print("/callback route:", status(ui + "/callback"), " (expect 200 via the SpaRewrite function, not 403)")
    print("root /         :", status(ui + "/"), " (expect 200)")
    # The authorize URL the app builds (challenge omitted here; the app derives it live):
    redirect = urllib.parse.quote(ui + "/callback", safe="")
    authorize = (f"{domain}/oauth2/authorize?response_type=code&client_id={client}"
                 f"&redirect_uri={redirect}&scope=openid+profile"
                 f"&code_challenge=<S256>&code_challenge_method=S256")
    print("\nauthorize URL the app redirects to:\n", authorize)
else:
    print("No deployment reachable — expected when deployed:")
    print("  /callback -> 200 (SpaRewrite serves /callback/index.html, ?code preserved)")
    print("  the authorize URL is <domain>/oauth2/authorize?response_type=code&client_id=...")
    print("    &redirect_uri=<ui>/callback&scope=openid+profile&code_challenge=<S256>&code_challenge_method=S256")


## What just changed

A banker can now sign in to the deployed UI, and you can explain every step. The app uses
authorization-code with PKCE because it is a public client with no secret, and the verifier
binding is what makes that safe. The hosted UI returns the user to `/callback?code=...`, and
a viewer-request CloudFront Function serves the right per-route document while preserving
the query string, so the code survives to the token exchange. Both pieces are the genuine
design: PKCE is the correct flow for a secretless SPA, and the path-preserving rewrite is
correct precisely because the naive 403-to-index fallback would drop the code.

This pairs with the deploy runbook in notebook 08, which handles the other half — the
two-pass callback-URL registration, where the first deploy creates the CloudFront
distributions and the second registers their `/callback` URLs on the Cognito app client so
the hosted UI will accept them. Notebook 08 teaches the deploy; this notebook teaches the
flow that deploy enables.

One honest framing to carry forward: this notebook teaches how the existing, deployed
system signs a user in. Proving it from nothing — a clean account following every notebook
end to end — is the job of the phase-3 end-to-end run, which exercises this exact path
rather than restating it here.
